# Logging my journey learning LLMs and PyTorch

It is long overdue to start learning PyTorch, and there is no better way to do it than by building a toy LLM. I've gone through a list of papers that helped me to understand how LLMs come to be, and I think it's a good idea to build a toy LLM to better connect the theory with practice.

# Starting the local colab docker image

Use the following command:
```
docker run -it --gpus=all -p 127.0.0.1:9000:8080 --mount type=bind,src="${HOME}/colab/checkpoints",dst=/content/checkpoints us-docker.pkg.dev/colab-images/public/runtime
```

In [1]:
!pwd && ls

/content
checkpoints  sample_data  tiktoken


# Setup PyTorch

In [2]:
import torch

In [3]:
print(f"PyTorch Version: {torch.__version__}")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
    print(f"Using GPU, CUDA version: {torch.version.cuda}, Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")

PyTorch Version: 2.10.0+cu128
Using GPU, CUDA version: 12.8, Number of GPUs: 1


# Get a working tokenizer
Let's use OpenAI's tiktoken tokenizer - alghough I am actually interested in building a minimum tokenizer from scratch later.

In [4]:
!git clone https://github.com/openai/tiktoken.git

fatal: destination path 'tiktoken' already exists and is not an empty directory.


In [5]:
import tiktoken

In [6]:
enc = tiktoken.get_encoding("r50k_base")

In [7]:
enc.encode("monkey d luffy")

[49572, 288, 300, 15352]

In [8]:
enc.decode(enc.encode("monkey d luffy"))

'monkey d luffy'

# Get the WebText dataset

In [9]:
from datasets import load_dataset

In [10]:
openwebtext = load_dataset("openwebtext")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

In [11]:
openwebtext

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8013769
    })
})

In [12]:
openwebtext['train']

Dataset({
    features: ['text'],
    num_rows: 8013769
})

## Test tokenizing the training data

In [13]:
enc.encode(openwebtext['train'][42]['text'])

[5159,
 8305,
 370,
 868,
 510,
 1165,
 1903,
 290,
 1719,
 2761,
 25446,
 736,
 284,
 3993,
 743,
 423,
 257,
 4633,
 2928,
 319,
 262,
 2612,
 11,
 257,
 2050,
 2523,
 198,
 198,
 8061,
 508,
 423,
 5876,
 38193,
 572,
 284,
 3993,
 743,
 307,
 379,
 3220,
 2526,
 286,
 2612,
 5287,
 11,
 4837,
 910,
 13,
 198,
 198,
 464,
 2050,
 11,
 3199,
 287,
 262,
 3427,
 8894,
 4913,
 11,
 3940,
 517,
 621,
 2026,
 11,
 830,
 661,
 329,
 1367,
 812,
 13,
 198,
 198,
 29193,
 1043,
 883,
 508,
 6989,
 1811,
 12513,
 286,
 3595,
 3993,
 547,
 517,
 1884,
 284,
 1205,
 262,
 4006,
 11,
 287,
 543,
 262,
 2612,
 10143,
 284,
 8901,
 6105,
 13,
 198,
 198,
 38897,
 910,
 2252,
 2267,
 318,
 2622,
 284,
 766,
 611,
 257,
 3092,
 286,
 3993,
 5640,
 2612,
 5287,
 393,
 262,
 2792,
 318,
 517,
 3716,
 13,
 198,
 198,
 1,
 42332,
 867,
 286,
 262,
 1243,
 326,
 4646,
 262,
 2863,
 286,
 2612,
 5287,
 635,
 4646,
 47104,
 26,
 922,
 5496,
 11,
 5517,
 11,
 3463,
 2994,
 290,
 407,
 9216,
 1583,
 5045,
 

# Embedding layer

In [14]:
vocabulary_size = enc.n_vocab
d_model = 768

embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)

Here we encode the phrase "monkey d luffy", which is then fed into the embedding_layer,
which produces d_model=768 embeddings, one 768 dimentional vector per token

In [15]:
embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

tensor([[[-0.4813,  0.2058, -0.4207,  ...,  1.1120,  0.7924, -0.9792],
         [-0.4535, -2.0964, -1.0627,  ..., -0.5576,  2.0410, -1.5004],
         [ 1.2976, -0.6852, -0.1947,  ..., -3.4904, -0.3331, -2.2429],
         [ 1.1290, -0.9621, -1.9864,  ...,  0.7277, -0.2006, -0.7525]]],
       grad_fn=<EmbeddingBackward0>)

In [16]:
embedding_layer.weight.shape

torch.Size([50257, 768])

Here we can see that the embedding layer is rather simple - it is a n_vocab x d_model matrix. So the embedding process is simply taking w[encoding].

## Positional Encoding

In [17]:
class LearnablePositionalEncoding(torch.nn.Module):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.pos_embeddings = torch.nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        # x is of shape (batch_size, seq_len, d_model)
        seq_len = x.size(1)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)

        pos_enc = self.pos_embeddings(position_ids)
        return x + pos_enc


In [18]:
pos_embed_layer = LearnablePositionalEncoding(1024, d_model)

In [19]:
pos_embed_layer(embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")])))

tensor([[[ 0.5402, -0.3558, -0.6569,  ...,  0.4440, -0.2610, -1.5797],
         [-1.4930, -1.5992, -1.3911,  ..., -1.4591,  2.4399, -1.1359],
         [ 2.1318,  0.4134,  0.1891,  ..., -3.1975, -0.7863, -1.7159],
         [ 2.4650, -0.6036, -2.4266,  ..., -0.1050,  2.2549,  0.0491]]],
       grad_fn=<AddBackward0>)

# Decoder Only Transformer Layer

This is the key part of the transformer architecture. Each attention layer is made of multi-head attention, normalization, and position-wise feedforward.

In GPT-2, normalization is moved at the beginning of each layer.

In [20]:
embedding = embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

## Layer Normalization

https://arxiv.org/pdf/1607.06450

In [21]:
layer_norm = torch.nn.LayerNorm(d_model)

In [22]:
x = layer_norm(embedding)

In [23]:
x

tensor([[[-0.4309,  0.2468, -0.3711,  ...,  1.1407,  0.8256, -0.9220],
         [-0.4366, -2.0470, -1.0337,  ..., -0.5387,  2.0087, -1.4628],
         [ 1.2496, -0.7605, -0.2632,  ..., -3.6041, -0.4035, -2.3396],
         [ 1.0097, -1.0640, -2.0798,  ...,  0.6118, -0.3088, -0.8561]]],
       grad_fn=<NativeLayerNormBackward0>)

In [24]:
x.shape

torch.Size([1, 4, 768])

## Multi-Head Attention

In [25]:
n_head = 12

# project into Q, K, V
projection = torch.nn.Linear(d_model, d_model * 3)

In [26]:
projection(x).shape

torch.Size([1, 4, 2304])

In [27]:
proj = projection(x).view(1, -1, 12, d_model // n_head * 3).permute(0, 2, 1, 3)

In [28]:
q, k, v = proj.chunk(3, dim=3)

In [29]:
import math
atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_model)

In [30]:
mask = torch.tril(torch.ones_like(atten[0][0]))

In [31]:
atten = atten.masked_fill(mask == 0, -float('inf'))

In [32]:
atten

tensor([[[[ 9.7807e-02,        -inf,        -inf,        -inf],
          [ 6.6068e-02,  9.4871e-02,        -inf,        -inf],
          [-7.6335e-03, -5.2055e-03, -1.3374e-02,        -inf],
          [ 1.2269e-01, -1.0379e-01, -5.5376e-02,  9.6611e-02]],

         [[-1.2075e-01,        -inf,        -inf,        -inf],
          [ 1.0620e-01, -1.6036e-01,        -inf,        -inf],
          [ 1.6287e-01, -2.3192e-02, -1.1512e-01,        -inf],
          [ 1.0781e-01,  4.6482e-02,  1.3484e-01, -5.2685e-02]],

         [[ 8.0824e-02,        -inf,        -inf,        -inf],
          [ 4.2113e-02, -1.8071e-01,        -inf,        -inf],
          [ 2.3612e-02,  2.1942e-03, -5.9571e-03,        -inf],
          [-2.1025e-01, -2.0520e-01,  4.8339e-02, -1.1461e-01]],

         [[-1.5616e-02,        -inf,        -inf,        -inf],
          [-2.0407e-01, -6.6031e-02,        -inf,        -inf],
          [ 1.3902e-01, -1.5642e-01,  3.0239e-02,        -inf],
          [-1.8393e-02,  8.4817e-0

In [33]:
softmax = torch.nn.Softmax(dim=-1)

atten = softmax(atten)

In [34]:
atten

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4928, 0.5072, 0.0000, 0.0000],
          [0.3337, 0.3345, 0.3318, 0.0000],
          [0.2771, 0.2210, 0.2319, 0.2700]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5662, 0.4338, 0.0000, 0.0000],
          [0.3865, 0.3209, 0.2927, 0.0000],
          [0.2618, 0.2462, 0.2690, 0.2230]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5555, 0.4445, 0.0000, 0.0000],
          [0.3390, 0.3318, 0.3291, 0.0000],
          [0.2272, 0.2284, 0.2943, 0.2501]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4655, 0.5345, 0.0000, 0.0000],
          [0.3786, 0.2818, 0.3396, 0.0000],
          [0.2430, 0.2694, 0.2150, 0.2726]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4743, 0.5257, 0.0000, 0.0000],
          [0.2938, 0.3589, 0.3473, 0.0000],
          [0.2676, 0.2436, 0.2739, 0.2149]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5676, 0.4324, 0.0000, 0.0000],
          [0.3207, 0.3

In [35]:
torch.matmul(atten, v).transpose(1, 2).reshape(1, 4, d_model)

tensor([[[ 0.9547, -0.4828, -0.1256,  ...,  0.6370,  0.3460,  0.8396],
         [ 0.5956, -0.4298,  0.2457,  ...,  0.5432,  0.2696,  0.2739],
         [ 0.4423, -0.1626,  0.3575,  ...,  0.1041,  0.1814, -0.2415],
         [ 0.2772,  0.0199,  0.1903,  ..., -0.1941,  0.0180, -0.2268]]],
       grad_fn=<UnsafeViewBackward0>)

### Putting it together

In [36]:
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        # project into K Q V
        self.input_linear = torch.nn.Linear(d_model, d_model * 3)
    def forward(self, x):
        batch_size, sequence_length, _ = x.size()
        proj = self.input_linear(x).view(batch_size, sequence_length, self.n_heads, self.d_model // self.n_heads * 3).permute(0, 2, 1, 3)
        q, k, v = proj.chunk(3, dim=-1)

        atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_model)
        mask = torch.tril(torch.ones_like(atten[0][0]))
        atten.masked_fill(mask == 0, -float('inf'))
        atten = F.softmax(atten, dim=-1)

        return torch.matmul(atten, v).transpose(1, 2).reshape(batch_size, sequence_length, d_model)


In [37]:
multi_head_attention = MultiHeadAttention(d_model, n_head)
multi_head_attention(x)

tensor([[[ 0.1461,  0.3825,  0.7400,  ..., -0.1312, -0.0070, -0.1315],
         [ 0.1653,  0.3565,  0.7520,  ..., -0.1430, -0.0399, -0.1274],
         [ 0.1528,  0.3547,  0.7260,  ..., -0.1252, -0.0635, -0.1076],
         [ 0.1564,  0.3729,  0.7568,  ..., -0.1188, -0.0064, -0.1221]]],
       grad_fn=<UnsafeViewBackward0>)

## Position-wise Feed Forward

In [38]:
d_hidden = 3072

In [39]:
class DenseFeedForward(torch.nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(d_hidden, d_model)
        )
    def forward(self, x):
        return self.model(x)

In [40]:
y = multi_head_attention(x)

In [41]:
dff = DenseFeedForward(d_model, d_hidden)

In [42]:
dff(y)

tensor([[[ 0.1637,  0.0351,  0.1179,  ...,  0.0642, -0.0208, -0.0556],
         [ 0.1702,  0.0293,  0.1048,  ...,  0.0643,  0.0064, -0.0578],
         [ 0.1665,  0.0339,  0.1143,  ...,  0.0655, -0.0030, -0.0603],
         [ 0.1629,  0.0222,  0.1237,  ...,  0.0560, -0.0022, -0.0552]]],
       grad_fn=<ViewBackward0>)

## The Full Transformer Layer

In [43]:
class TransformerDecoderOnly(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head):
        super().__init__()
        self.layer_norm_0 = torch.nn.LayerNorm(d_model)
        self.multi_head_attention = MultiHeadAttention(d_model, n_head)
        self.layer_norm_1 = torch.nn.LayerNorm(d_model)
        self.dff = DenseFeedForward(d_model, d_hidden)
    def forward(self, x):
        # layer normalization first
        x = self.layer_norm_0(x)

        # multi-head attention and residual
        identity = x
        x = self.multi_head_attention(x)
        x = x + identity

        # layer normalization before dense feed forward
        x = self.layer_norm_1(x)

        # dense feed forward and residual
        identity = x
        x = self.dff(x)
        return x + identity

In [44]:
transformer_decoder_only = TransformerDecoderOnly(d_model, d_hidden, n_head)

transformer_decoder_only(embedding)

tensor([[[-0.3189,  0.4432, -0.0442,  ...,  0.9825,  0.3315, -0.7610],
         [-0.4511, -1.6600, -0.4875,  ..., -0.7725,  1.2147, -1.0359],
         [ 1.1723, -0.8165,  0.1454,  ..., -3.5479, -0.5255, -2.3020],
         [ 1.0671, -0.7214, -1.7730,  ...,  0.4702, -0.7287, -0.6536]]],
       grad_fn=<AddBackward0>)

# Output Layer

The output layer is rather simple. matrix multiply with the input embedding matrix, and soft max. We can then reverse the tokenization. 

In [45]:
torch.matmul(transformer_decoder_only(embedding), embedding_layer.weight.transpose(0, 1))

tensor([[[-40.0242,  12.1333, -20.4647,  ..., -22.1519, -20.7269,  25.9752],
         [-28.3820,   4.6567,   1.3226,  ...,   3.0233,   4.6906,  -2.9469],
         [-24.9005,  40.9949,  36.3794,  ...,  -9.7722,   0.2179,  21.6060],
         [ 33.0758,  36.2687,  -7.9311,  ...,  -5.0288,  24.5138,   8.2036]]],
       grad_fn=<UnsafeViewBackward0>)

In [46]:
class Output(torch.nn.Module):
    def __init__(self, embedding_layer_weights):
        super().__init__()
        self.layer_norm = torch.nn.LayerNorm(embedding_layer_weights.shape[1])
        self.embedding_layer_weights = embedding_layer_weights
    
    def forward(self, x):
        # x is of shape (batch_size, seq_length, d_model)
        # self.embedding_layer_weights is of shape (vocab_size, d_model)
        x = self.layer_norm(x)
        return torch.matmul(x, self.embedding_layer_weights.transpose(0, 1))

# The full model

In [47]:
class ToyGPT(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head, vocab_size, layers, max_seq_len):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)
        self.pos_encoding_layer = LearnablePositionalEncoding(max_seq_len, d_model)
        self.transformers = torch.nn.Sequential()
        for i in range (0, layers):
            self.transformers.append(TransformerDecoderOnly(d_model, d_hidden, n_head))
        self.output_layer = Output(self.embedding_layer.weight)
    
    def forward(self, x):
        x = self.embedding_layer(x)
        x = self.pos_encoding_layer(x)
        x = self.transformers(x)
        return self.output_layer(x)


# Training

In [48]:
batch_size = 48
max_seq_len = 1024

In [49]:
enc._special_tokens

{'<|endoftext|>': 50256}

Pre-tokenize the training data to avoid CPU bottleneck.

In [50]:
from datasets import Dataset

rows = openwebtext["train"].num_rows
bos_token = '<|endoftext|>'

def tokenize(examples):
    out = enc.encode(examples['text']+ bos_token, allowed_special={bos_token})
    return {'tokenized_text': out}

tokenized_data = openwebtext["train"].map(tokenize, remove_columns=openwebtext["train"].column_names, num_proc=32)


In [51]:
def do_checkpoint(model, optimizer, running_encoding, row_number):
    torch.save({
        'row_number': row_number,
        'running_encoding': running_encoding,
        'optimizer_state_dict': optimizer.state_dict(),
        'model_state_dict': model.state_dict(),
    }, 'checkpoints/check_point_{}'.format(row_number))

In [61]:
import gc
from tqdm.notebook import tqdm

def train(start_row_number = 0, check_point_interval = 100000, summary_writer = None):
    tokens_per_batch = (batch_size + 1) * max_seq_len // 2

    encoding = []

    model = ToyGPT(d_model, d_hidden, n_head, enc.n_vocab, 12, max_seq_len)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0006, weight_decay=0.01)
    model.to(device)
    model.train()

    model = torch.compile(model)

    if start_row_number != 0:
        checkpoint = torch.load('checkpoints/check_point_{}'.format(start_row_number))

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        encoding = checkpoint['running_encoding']
    
    write_summary_threshold = start_row_number
    check_point_threshold = (start_row_number + check_point_interval) // check_point_interval * check_point_interval

    data = tokenized_data.select(range(start_row_number, tokenized_data.num_rows)).to_iterable_dataset(num_shards=8)


    for row in enumerate(tqdm(data, total = rows, initial = start_row_number)):
        encoding.extend([enc._special_tokens[bos_token]] + row[1]['tokenized_text'])
        cur_row_number = start_row_number + row[0]
        if len(encoding) > tokens_per_batch:
            residual = encoding[tokens_per_batch:]
            encoding = encoding[0:tokens_per_batch + 1]
            batch = None
            target = None
            for i in range(0, batch_size):
              batch_row = torch.LongTensor(encoding[i * max_seq_len // 2 : i * max_seq_len // 2 + max_seq_len])
              batch_row = batch_row.to(device)
              target_row = torch.LongTensor(encoding[i * max_seq_len // 2 + 1 : i * max_seq_len // 2 + max_seq_len + 1])
              target_row = target_row.to(device)
              if batch is None:
                  batch = batch_row
                  target = target_row
              else:
                  batch = torch.cat((batch, batch_row), dim=0)
                  target = torch.cat((target, target_row), dim=0)

            batch = batch.to(device)
            batch = batch.reshape(batch_size, max_seq_len)
            target = target.to(device)
            target = target.reshape(batch_size, max_seq_len)
    
            optimizer.zero_grad()
    
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                out = model(batch)
                loss = F.cross_entropy(out.view(-1, enc.n_vocab), target.view(-1))
    
            loss.backward()
            optimizer.step()
    
            if summary_writer and cur_row_number >= write_summary_threshold:
                write_summary_threshold = (write_summary_threshold + 1000) // 1000 * 1000
                summary_writer.add_scalar('Loss/train', loss.item(), cur_row_number)

            if (cur_row_number + 1 >= check_point_threshold):
                print("check pointing at {}".format(cur_row_number + 1))
                do_checkpoint(model, optimizer, encoding, cur_row_number + 1)
                check_point_threshold = (cur_row_number + 1 + check_point_interval) // check_point_interval * check_point_interval
    
            encoding = residual
    
            gc.collect()

In [ ]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:
from torch.utils.tensorboard import SummaryWriter

summary_writer = SummaryWriter('./adamw_run_lr_0006')

In [ ]:
%tensorboard --logdir adamw_run_lr_0006


In [ ]:
train(start_row_number=0, check_point_interval=100000, summary_writer = summary_writer)

NameError: name 'train' is not defined